# 05 — Executive Dashboard: Project Profitability & Utilisation Analytics
Renders an in-notebook HTML dashboard using `displayHTML()` with KPI cards and summary tables.

In [ ]:
LAKEHOUSE_NAME = "proservices_lakehouse"
GOLD  = "gold"
VIEWS = "reporting"

from pyspark.sql import functions as F

In [ ]:
# ── Pull KPIs ────────────────────────────────────────────────────────────────
kpi = spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{VIEWS}.vw_executive_summary").collect()[0]

total_engagements = f"{kpi['total_engagements']:,}"
avg_margin        = f"{kpi['avg_margin_pct']:.1f}%"
at_risk           = f"{kpi['at_risk_engagements']:,}"
total_revenue     = f"£{int(kpi['total_revenue_gbp']):,}"

# Avg NPS from clients
avg_nps = spark.table(f"{LAKEHOUSE_NAME}.{GOLD}.gold_client_revenue") \
    .agg(F.round(F.avg("nps_score"), 0).alias("n")).collect()[0]["n"]
nps_display = f"{int(avg_nps)}"

# Overall utilisation rate
util = spark.table(f"{LAKEHOUSE_NAME}.{GOLD}.gold_consultant_utilisation") \
    .agg(F.round(F.avg("utilisation_rate_pct"), 1).alias("u")).collect()[0]["u"]
util_display = f"{util:.1f}%"

print("KPIs collected.")

In [ ]:
# ── Low-margin projects table ────────────────────────────────────────────────
lm_rows = (
    spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{VIEWS}.vw_low_margin_projects")
    .limit(8).collect()
)

lm_html = ""
for r in lm_rows:
    mc = "#d32f2f" if r["margin_pct"] < 0 else "#f57c00" if r["margin_pct"] < 5 else "#388e3c"
    lm_html += f"""<tr>
        <td>{r['engagement_id']}</td>
        <td>{r['practice']}</td>
        <td>{r['status']}</td>
        <td>£{r['budget_gbp']:,.0f}</td>
        <td style='color:{mc};font-weight:bold'>{r['margin_pct']:.1f}%</td>
        <td>{r['overrun_pct']:.1f}%</td>
    </tr>"""

In [ ]:
# ── Consultant utilisation table ─────────────────────────────────────────────
con_rows = (
    spark.table(f"{LAKEHOUSE_NAME}.{GOLD}.gold_consultant_utilisation")
    .orderBy(F.col("utilisation_rate_pct").desc())
    .limit(8).collect()
)

con_html = ""
for r in con_rows:
    uc = "#388e3c" if r["utilisation_rate_pct"] >= 75 else "#f57c00" if r["utilisation_rate_pct"] >= 60 else "#d32f2f"
    con_html += f"""<tr>
        <td>{r['consultant_id']}</td>
        <td>{r['grade']}</td>
        <td>{r['practice']}</td>
        <td style='color:{uc};font-weight:bold'>{r['utilisation_rate_pct']:.1f}%</td>
        <td>{r['billable_hours']:.0f} hrs</td>
        <td>£{r['total_billed_gbp']:,.0f}</td>
    </tr>"""

In [ ]:
# ── Render Dashboard ─────────────────────────────────────────────────────────
html = f"""
<style>
  body{{font-family:'Segoe UI',Arial,sans-serif;background:#f5f5f5;padding:24px;}}
  h1{{color:#1565c0;font-size:1.6em;margin-bottom:4px;}}
  .subtitle{{color:#666;margin-bottom:20px;font-size:.9em;}}
  .kpi-grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:16px;margin-bottom:28px;}}
  .kpi{{background:#fff;border-radius:10px;padding:18px 20px;box-shadow:0 2px 8px rgba(0,0,0,.07);}}
  .kpi .label{{font-size:.78em;color:#888;text-transform:uppercase;letter-spacing:.05em;}}
  .kpi .value{{font-size:1.9em;font-weight:700;color:#1565c0;margin-top:4px;}}
  .section{{background:#fff;border-radius:10px;padding:20px;box-shadow:0 2px 8px rgba(0,0,0,.07);margin-bottom:24px;}}
  .section h3{{margin:0 0 14px;color:#333;font-size:1em;}}
  table{{width:100%;border-collapse:collapse;font-size:.88em;}}
  th{{background:#1565c0;color:#fff;padding:8px 12px;text-align:left;font-weight:600;}}
  td{{padding:7px 12px;border-bottom:1px solid #eee;}}
  tr:hover td{{background:#f0f4ff;}}
</style>
<h1>📋 Project Profitability & Utilisation Analytics</h1>
<div class='subtitle'>Professional Services | Portfolio Performance</div>

<div class='kpi-grid'>
  <div class='kpi'><div class='label'>Total Engagements</div><div class='value'>{total_engagements}</div></div>
  <div class='kpi'><div class='label'>Avg Project Margin</div><div class='value' style='color:#388e3c'>{avg_margin}</div></div>
  <div class='kpi'><div class='label'>Billable Utilisation</div><div class='value'>{util_display}</div></div>
  <div class='kpi'><div class='label'>Total Revenue</div><div class='value' style='color:#388e3c'>{total_revenue}</div></div>
  <div class='kpi'><div class='label'>At-Risk Engagements</div><div class='value' style='color:#d32f2f'>{at_risk}</div></div>
  <div class='kpi'><div class='label'>Avg Client NPS</div><div class='value'>{nps_display}</div></div>
</div>

<div class='section'>
  <h3>🔴 Low-Margin Projects (margin &lt; 5%)</h3>
  <table>
    <tr><th>Engagement</th><th>Practice</th><th>Status</th><th>Budget</th><th>Margin %</th><th>Overrun %</th></tr>
    {lm_html}
  </table>
</div>

<div class='section'>
  <h3>👤 Top Consultants by Utilisation</h3>
  <table>
    <tr><th>Consultant</th><th>Grade</th><th>Practice</th><th>Utilisation</th><th>Billable Hours</th><th>Revenue</th></tr>
    {con_html}
  </table>
</div>
"""

displayHTML(html)